# Gene Transformer Training

ScipherModel = GeneTransformerEmbedder + WideNN with MarginalizationLoss on blood cell hierarchy (CL:0000988).

**Runs on cluster** (needs SOMA database + ESM-2 embeddings).

## 1. Setup & Data Loading

In [ ]:
import sys
import logging
import pickle
import time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils import clip_grad_norm_
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt

import tiledbsoma as soma
from tiledbsoma_ml import ExperimentDataset, experiment_dataloader

# Find project root by searching upward for pyproject.toml
_cwd = Path(".").resolve()
PROJECT_ROOT = _cwd
for _parent in [_cwd] + list(_cwd.parents):
    if (_parent / "pyproject.toml").exists():
        PROJECT_ROOT = _parent
        break
sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / "data"
print(f"Project root: {PROJECT_ROOT}")

from scipher.hierarchy import (
    load_prebuilt_hierarchy, MarginalizationLoss, WideNN,
)
from scipher.embedders.gene_transformer import GeneTransformerEmbedder

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

# --- Config ---
HIERARCHY_DATE = "2026-01-29"  # date of preprocessed hierarchy artifacts
ROOT_CL_ID = "CL:0000988"  # blood cells
SOMA_URI = "/scratch/sigbio_project_root/sigbio_project25/jingqiao/mccell-single/soma_db_homo_sapiens"
MIN_CELL_COUNT = 50
BATCH_SIZE = 64  # per GPU
LR = 1e-4
LEAF_WEIGHT = 7.0
GRAD_CLIP = 1.0
EPOCHS = 10
SEED = 42
NUM_WORKERS = 3  # 4 CPUs: leave 1 for main process

# GeneTransformerEmbedder hyperparameters
D_MODEL = 512
N_LAYERS = 4
N_HEADS = 8
N_CLS = 8       # multi-CLS readout tokens
D_FF = 2048
OUTPUT_DIM = 512
DROPOUT = 0.1

# Checkpoint directory
cl_folder = ROOT_CL_ID.replace(":", "")
run_date = datetime.now().strftime("%Y-%m-%d")
checkpoint_dir = DATA_DIR / "checkpoint" / f"transformer_{run_date}_{HIERARCHY_DATE}_{cl_folder}"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
num_gpus = torch.cuda.device_count()

# DataParallel splits the DataLoader batch across GPUs, so we scale up
effective_batch_size = BATCH_SIZE * max(num_gpus, 1)

print(f"Device: {device}")
print(f"Checkpoints: {checkpoint_dir}")
if torch.cuda.is_available():
    print(f"GPUs: {num_gpus}x {torch.cuda.get_device_name(0)}")
    print(f"Batch size: {BATCH_SIZE}/GPU x {num_gpus} GPUs = {effective_batch_size} effective")
print(f"DataLoader workers: {NUM_WORKERS}")
print(f"GeneTransformer: d_model={D_MODEL}, n_layers={N_LAYERS}, n_heads={N_HEADS}, "
      f"n_cls={N_CLS}, d_ff={D_FF}, output_dim={OUTPUT_DIM}")

In [ ]:
(
    mapping_dict, leaf_values, internal_values,
    marginalization_df, parent_child_df, exclusion_df,
) = load_prebuilt_hierarchy(HIERARCHY_DATE, ROOT_CL_ID)

all_cell_values = list(leaf_values) + list(internal_values)
print(f"Leaf types: {len(leaf_values)}, Internal: {len(internal_values)}, Total: {len(all_cell_values)}")

In [ ]:
EMB_PATH = DATA_DIR / "embeddings" / "gene_to_embedding.pkl"
with open(EMB_PATH, "rb") as f:
    gene_to_embedding = pickle.load(f)

embed_dim = next(iter(gene_to_embedding.values())).shape[0]
print(f"Gene embeddings: {len(gene_to_embedding):,} genes x {embed_dim}-dim")

In [ ]:
BIOMART_FILE = DATA_DIR / "raw" / "mart_export.txt"
df_biomart = pd.read_csv(BIOMART_FILE)
df_pc = df_biomart[df_biomart["Gene type"] == "protein_coding"].dropna(
    subset=["Gene stable ID", "Gene name"]
)
gene_list = df_pc["Gene stable ID"].unique().tolist()

# Build Ensembl ID -> gene symbol mapping for remapping streamed var columns
ensembl_to_symbol = (
    df_pc.drop_duplicates("Gene stable ID")
    .set_index("Gene stable ID")["Gene name"]
    .to_dict()
)
print(f"Protein-coding Ensembl IDs: {len(gene_list):,}")
print(f"Ensembl->symbol mappings: {len(ensembl_to_symbol):,}")

In [ ]:
experiment = soma.open(SOMA_URI, mode="r")

# Step 1: Read obs metadata (lightweight, no expression data)
obs_df = (
    experiment.obs.read(
        value_filter='assay == "10x 3\' v3" and is_primary_data == True',
        column_names=["soma_joinid", "cell_type_ontology_term_id", "cell_type"],
    )
    .concat()
    .to_pandas()
)
print(f"Total 10x v3 primary cells: {len(obs_df):,}")

# Filter to blood cell hierarchy types only
obs_df = obs_df[obs_df["cell_type_ontology_term_id"].isin(all_cell_values)].copy()
print(f"Blood cells in hierarchy: {len(obs_df):,}")

# Drop rare cell types
type_counts = obs_df["cell_type_ontology_term_id"].value_counts()
keep_types = type_counts[type_counts >= MIN_CELL_COUNT].index
obs_df = obs_df[obs_df["cell_type_ontology_term_id"].isin(keep_types)].copy()
print(
    f"After dropping < {MIN_CELL_COUNT} cells: {len(obs_df):,} cells, "
    f"{obs_df['cell_type_ontology_term_id'].nunique()} types"
)

# Build cl_to_name mapping from obs metadata (before we discard it)
cl_to_name = (
    obs_df.drop_duplicates("cell_type_ontology_term_id")
    .set_index("cell_type_ontology_term_id")["cell_type"]
    .to_dict()
)

# Step 2: Create streaming datasets via ExperimentDataset
joinids = obs_df["soma_joinid"].values
var_value_filter = f"feature_id in {gene_list}"

with experiment.axis_query(
    measurement_name="RNA",
    obs_query=soma.AxisQuery(coords=(joinids,)),
    var_query=soma.AxisQuery(value_filter=var_value_filter),
) as query:
    # Read var metadata to know which columns map to which genes
    var_df = query.var(column_names=["feature_id", "feature_name"]).concat().to_pandas()

    # Create streaming dataset (extracts x_locator + query_ids internally)
    ds = ExperimentDataset(
        query,
        layer_name="raw",
        obs_column_names=["cell_type_ontology_term_id"],
        batch_size=effective_batch_size,
        shuffle=True,
        seed=SEED,
    )

# Context is closed; ds holds serializable x_locator + query_ids
# Train/val split
train_ds, val_ds = ds.random_split(0.8, 0.2, seed=SEED)

# Val set should not shuffle
from attr import evolve
val_ds = evolve(val_ds, shuffle=False)

train_loader = experiment_dataloader(train_ds, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
val_loader = experiment_dataloader(val_ds, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

n_obs, n_vars = ds.shape
print(f"\nStreaming dataset: {n_obs:,} cells x {n_vars:,} genes")
print(f"Train batches: ~{train_ds.shape[0]:,}, Val batches: ~{val_ds.shape[0]:,}")

# Step 3: Build gene embedding index from var metadata
col_indices = []       # positional indices into X columns that have embeddings
gene_names_mapped = [] # corresponding gene symbols

seen_symbols = set()
for pos, (_, row) in enumerate(var_df.iterrows()):
    ensembl_id = row["feature_id"]
    symbol = ensembl_to_symbol.get(ensembl_id, row["feature_name"])
    if symbol in gene_to_embedding and symbol not in seen_symbols:
        col_indices.append(pos)
        gene_names_mapped.append(symbol)
        seen_symbols.add(symbol)

col_indices = np.array(col_indices)

# Build gene embedding tensor (loaded once, shared across all batches)
gene_embs_tensor = torch.stack(
    [torch.from_numpy(gene_to_embedding[g]) for g in gene_names_mapped]
).to(device)  # (n_mapped_genes, embed_dim)

print(f"Genes with embeddings: {len(col_indices):,}/{len(var_df):,} ({100*len(col_indices)/len(var_df):.1f}%)")
print(f"Gene embedding tensor: {gene_embs_tensor.shape}")

In [ ]:
# Reverse mappings for evaluation (no adata dependency)
idx_to_cl = {v: k for k, v in mapping_dict.items()}
leaf_idx_set = {mapping_dict[cl] for cl in leaf_values}

## 2. Model & Training

In [ ]:
class ScipherModel(nn.Module):
    def __init__(self, embed_dim, num_leaves, gene_embs,
                 d_model=512, n_layers=4, n_heads=8, n_cls=8, d_ff=2048, output_dim=512, dropout=0.1):
        super().__init__()
        self.embedder = GeneTransformerEmbedder(
            gene_embed_dim=embed_dim, d_model=d_model, output_dim=output_dim,
            n_layers=n_layers, n_heads=n_heads, n_cls=n_cls, d_ff=d_ff, dropout=dropout,
        )
        self.classifier = WideNN(input_dim=output_dim, output_dim=num_leaves)
        # Register as buffer so DataParallel replicates (not scatters) to each GPU
        self.register_buffer("gene_embs", gene_embs)

    def forward(self, expression, mask):
        cell_embedding, attn_weights = self.embedder(
            self.gene_embs, expression, mask,
        )
        logits = self.classifier(cell_embedding)
        return logits, cell_embedding, attn_weights

In [ ]:
model = ScipherModel(
    embed_dim=embed_dim, num_leaves=len(leaf_values),
    gene_embs=gene_embs_tensor,
    d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS,
    n_cls=N_CLS, d_ff=D_FF, output_dim=OUTPUT_DIM, dropout=DROPOUT,
).to(device)
if num_gpus > 1:
    model = nn.DataParallel(model)
    print(f"Using DataParallel across {num_gpus} GPUs")

loss_fn = MarginalizationLoss(
    marginalization_df, parent_child_df, exclusion_df,
    leaf_values, internal_values, mapping_dict,
    leaf_weight=LEAF_WEIGHT, device=device,
)
optimizer = optim.Adam(model.parameters(), lr=LR)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"ScipherModel (Transformer): {n_params:,} trainable parameters")
emb_params = sum(p.numel() for n, p in model.named_parameters() if "embedder" in n)
cls_params = sum(p.numel() for n, p in model.named_parameters() if "classifier" in n)
print(f"  GeneTransformerEmbedder: {emb_params:,}")
print(f"  WideNN classifier: {cls_params:,}")

In [ ]:
loss_history = {"total": [], "leaf": [], "parent": []}
epoch_times = []

print(f"Training: {EPOCHS} epochs")
print(f"Saving checkpoints to: {checkpoint_dir}")
print("-" * 70)

for epoch in range(EPOCHS):
    model.train()
    train_ds.set_epoch(epoch)
    epoch_start = time.time()
    epoch_losses = []

    for i, (X_batch, obs_batch) in enumerate(train_loader):
        X = torch.from_numpy(X_batch) if isinstance(X_batch, np.ndarray) else torch.from_numpy(X_batch.toarray())
        X = X.float()

        X_mapped = X[:, col_indices]
        mask = (X_mapped > 0).to(device)

        # Expression preprocessing: log1p + L1 normalization
        X_log = torch.log1p(X_mapped)
        expr_sum = X_log.sum(dim=1, keepdim=True).clamp(min=1e-10)
        expression = (X_log / expr_sum).to(device)

        labels = obs_batch["cell_type_ontology_term_id"].values
        y_batch = torch.tensor(
            [mapping_dict[t] for t in labels], device=device, dtype=torch.long
        )

        optimizer.zero_grad()
        logits, _, _ = model(expression, mask)
        total_loss, loss_leafs, loss_parents = loss_fn(logits, y_batch)
        total_loss.backward()
        clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()

        loss_history["total"].append(total_loss.item())
        loss_history["leaf"].append(loss_leafs.item())
        loss_history["parent"].append(loss_parents.item())
        epoch_losses.append(total_loss.item())

        if (i + 1) % 200 == 0:
            avg_recent = np.mean(epoch_losses[-200:])
            print(
                f"  [Epoch {epoch+1}/{EPOCHS}, Batch {i+1}] "
                f"Loss: {total_loss.item():.4f} (avg: {avg_recent:.4f})"
            )

    epoch_time = time.time() - epoch_start
    epoch_times.append(epoch_time)
    avg_loss = np.mean(epoch_losses)

    state_dict = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
    ckpt = {
        "epoch": epoch + 1,
        "model_state_dict": state_dict,
        "optimizer_state_dict": optimizer.state_dict(),
        "loss_history": {k: list(v) for k, v in loss_history.items()},
        "epoch_times": list(epoch_times),
        "config": {
            "input_dim": embed_dim, "output_dim": OUTPUT_DIM,
            "run_date": run_date, "hierarchy_date": HIERARCHY_DATE,
            "root_cl_id": ROOT_CL_ID, "model_class": "ScipherModel",
            "embedder_class": "GeneTransformerEmbedder",
            "d_model": D_MODEL, "n_layers": N_LAYERS, "n_heads": N_HEADS,
            "n_cls": N_CLS, "d_ff": D_FF, "dropout": DROPOUT,
            "lr": LR, "batch_size": BATCH_SIZE, "leaf_weight": LEAF_WEIGHT,
            "num_gpus": num_gpus,
        },
    }
    ckpt_path = checkpoint_dir / f"epoch{epoch+1:02d}.pt"
    torch.save(ckpt, ckpt_path)

    print(
        f"Epoch {epoch+1}/{EPOCHS} -- avg loss: {avg_loss:.4f}, "
        f"time: {epoch_time:.1f}s, saved: {ckpt_path.name}"
    )

## 3. Evaluation

In [ ]:
model.eval()
all_preds, all_labels = [], []
val_cell_embeddings = []
val_cell_types = []

with torch.no_grad():
    for X_batch, obs_batch in val_loader:
        X = torch.from_numpy(X_batch) if isinstance(X_batch, np.ndarray) else torch.from_numpy(X_batch.toarray())
        X = X.float()

        X_mapped = X[:, col_indices]
        mask = (X_mapped > 0).to(device)
        X_log = torch.log1p(X_mapped)
        expr_sum = X_log.sum(dim=1, keepdim=True).clamp(min=1e-10)
        expression = (X_log / expr_sum).to(device)

        labels = obs_batch["cell_type_ontology_term_id"].values
        y_batch = torch.tensor(
            [mapping_dict[t] for t in labels], device=device, dtype=torch.long
        )

        logits, cell_emb, _ = model(expression, mask)
        preds = torch.argmax(logits, dim=1)

        is_leaf = torch.tensor(
            [y.item() in leaf_idx_set for y in y_batch], device=device
        )
        if is_leaf.sum() > 0:
            all_preds.extend(preds[is_leaf].cpu().numpy())
            all_labels.extend(y_batch[is_leaf].cpu().numpy())

        val_cell_embeddings.append(cell_emb.cpu().numpy())
        val_cell_types.extend(
            cl_to_name.get(t, t) for t in labels
        )

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
val_embeddings = np.concatenate(val_cell_embeddings, axis=0)

acc = (all_preds == all_labels).mean()
macro_f1 = f1_score(all_labels, all_preds, average="macro")
weighted_f1 = f1_score(all_labels, all_preds, average="weighted")

print(f"=== Validation (leaf-labeled cells) ===")
print(f"  Leaf accuracy: {acc:.4f}")
print(f"  Macro F1:      {macro_f1:.4f}")
print(f"  Weighted F1:   {weighted_f1:.4f}")
print(f"  N samples:     {len(all_labels):,}")

# Per-class accuracy
rows = []
for idx in sorted(set(all_labels)):
    cl_id = idx_to_cl[idx]
    name = cl_to_name.get(cl_id, cl_id)
    mask_idx = all_labels == idx
    n = mask_idx.sum()
    class_acc = (all_preds[mask_idx] == all_labels[mask_idx]).mean()
    rows.append({"cell_type": name, "cl_id": cl_id, "n": n, "accuracy": class_acc})

df_eval = pd.DataFrame(rows).sort_values("accuracy")
print(f"\nPer-class accuracy:")
print(df_eval.to_string(index=False, float_format="{:.3f}".format))

In [ ]:
print("=" * 60)
print("Training Summary")
print("=" * 60)
print(f"  Embedder:      GeneTransformerEmbedder")
print(f"  Leaf accuracy: {acc:.4f}")
print(f"  Macro F1:      {macro_f1:.4f}")
print(f"  Weighted F1:   {weighted_f1:.4f}")
print(f"  Epochs:        {EPOCHS}")
print(f"  Total time:    {sum(epoch_times):.1f}s")
print(f"  Checkpoints:   {checkpoint_dir}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

def smooth(values, window=50):
    kernel = np.ones(window) / window
    return np.convolve(values, kernel, mode="valid")

for ax, key, title in zip(
    axes, ["total", "leaf", "parent"],
    ["Total Loss", "Leaf Loss", "Parent Loss"],
):
    if len(loss_history[key]) > 50:
        ax.plot(smooth(loss_history[key]), alpha=0.8)
    else:
        ax.plot(loss_history[key], alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel("Batch")
    ax.set_ylabel("Loss")

plt.tight_layout()
plt.savefig(DATA_DIR / "reports" / "transformer_training_loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to data/reports/transformer_training_loss_curves.png")

In [ ]:
import scanpy as sc

sc.settings.set_figure_params(dpi=100, frameon=False)

# Build AnnData for UMAP visualization from collected val embeddings
adata_umap = sc.AnnData(
    X=np.zeros((len(val_cell_types), 1)),  # placeholder
    obs=pd.DataFrame({"cell_type": val_cell_types}),
)
adata_umap.obsm["X_transformer"] = val_embeddings

sc.pp.neighbors(adata_umap, use_rep="X_transformer")
sc.tl.umap(adata_umap)

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
sc.pl.umap(
    adata_umap, color="cell_type", ax=ax, show=False,
    title="Gene Transformer Cell Embeddings",
    legend_loc="on data", legend_fontsize=6, frameon=False,
)

plt.tight_layout()
plt.savefig(
    DATA_DIR / "reports" / "transformer_train_embedding_umap.png",
    dpi=150, bbox_inches="tight",
)
plt.show()
print("Saved to data/reports/transformer_train_embedding_umap.png")